# Grounded QA Validation Notebook

This notebook validates synthetic question-answer pairs against **cited source PDFs and page ranges**.

Goal: flag every potentially hallucinated pair and export only rows that are grounded in the source evidence.

Validation mode implemented here:
- Deterministic grounding checks from cited pages
- Optional LLM entailment adjudication for uncertain rows
- Output artifacts:
  - row-level pass/fail report with reason codes
  - evidence snippets with page spans
  - clean filtered dataset export

## 1. Set Up the Project Structure

Create paths, import dependencies, and define helper constants used by the validator.

In [36]:
!pip install pymupdf pandas tqdm openai python-dotenv

In [37]:
# If needed in a fresh environment, uncomment:
import json
import logging
import os
import re
import unicodedata
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import fitz
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler()],
)
logger = logging.getLogger("qa-grounding-validator")

ROOT = Path.cwd().resolve()
DEFAULT_BOOKS_DIR = ROOT.parents[1] / "datasets" / "training" / "Books"
print(DEFAULT_BOOKS_DIR)
DEFAULT_OUTPUT_DIR = ROOT / "outputs" / "qa_validation"
DEFAULT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANDATORY_FIELDS = ["question", "answer", "book_name", "page_start", "page_end"]

logger.info("Setup complete.")

2026-04-08 20:26:03,694 | INFO | Setup complete.


/media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/datasets/training/Books


## 2. Implement Core Data Models

Define canonical schema objects and runtime configuration for deterministic + hybrid validation.

In [38]:
!env | grep OPENAI

OPENAI_LLM_API_URL=https://llm.ipropel.co.in/v1
OPENAI_LLM_MODEL_NAME=openai/gpt-oss-20b


In [39]:
@dataclass
class RuntimeConfig:
    dataset_path: Path
    books_dir: Path = DEFAULT_BOOKS_DIR
    output_dir: Path = DEFAULT_OUTPUT_DIR
    checkpoint_path: Optional[Path] = None

    # Validation strictness
    lexical_threshold: float = 0.22
    claim_support_threshold: float = 0.18
    weak_support_floor: float = 0.12

    # LLM judge options
    enable_llm_judge: bool = True
    judge_on_uncertain_only: bool = True
    judge_model: str = os.getenv("OPENAI_LLM_MODEL_NAME", "openai/gpt-oss-20b")
    judge_base_url: str = os.getenv("OPENAI_LLM_API_URL", "")
    judge_api_key: str = os.getenv("OPENAI_LLM_API_KEY", "")
    judge_confidence_threshold: float = 0.72

    # Retrieval options
    max_evidence_chars: int = 14000
    max_rows: Optional[int] = None  # set to an int for smoke testing

    def __post_init__(self):
        self.dataset_path = Path(self.dataset_path)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        if self.checkpoint_path is None:
            self.checkpoint_path = self.output_dir / "validation_checkpoint.json"


@dataclass
class DeterministicResult:
    lexical_score: float
    claim_support_score: float
    contradiction: bool
    contradiction_reasons: List[str]
    verdict: str
    reason_code: str
    best_support_sentence: str
    matched_pages: List[int]


@dataclass
class JudgeResult:
    verdict: str
    confidence: float
    reason_code: str
    cited_spans: List[str]


logger.info("Core data models defined.")

2026-04-08 20:26:03,859 | INFO | Core data models defined.


## 3. Build the First Functional Module

Implement schema normalization, PDF page evidence extraction, deterministic grounding checks, optional LLM adjudication, and end-to-end validation orchestration.

In [40]:
def normalize_text(text: str) -> str:
    text = text or ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def sentence_split(text: str) -> List[str]:
    text = normalize_text(text)
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip() for p in parts if p.strip()]


def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9_\-]+", normalize_text(text).lower())


def jaccard_score(a: str, b: str) -> float:
    ta, tb = set(tokenize(a)), set(tokenize(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def parse_page_range(value: Any) -> Tuple[Optional[int], Optional[int]]:
    if value is None:
        return None, None
    if isinstance(value, (list, tuple)) and len(value) >= 2:
        try:
            s, e = int(value[0]), int(value[1])
            return min(s, e), max(s, e)
        except Exception:
            return None, None

    s = str(value).strip()
    if not s:
        return None, None

    nums = [int(x) for x in re.findall(r"\d+", s)]
    if len(nums) >= 2:
        a, b = nums[0], nums[1]
        return min(a, b), max(a, b)
    if len(nums) == 1:
        return nums[0], nums[0]
    return None, None


def safe_json_load(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return pd.DataFrame(rows)
    if suffix == ".json":
        payload = safe_json_load(path)
        if isinstance(payload, list):
            return pd.DataFrame(payload)
        if isinstance(payload, dict):
            # Try common wrappers
            for k in ("data", "rows", "records", "items"):
                if isinstance(payload.get(k), list):
                    return pd.DataFrame(payload[k])
            return pd.DataFrame([payload])
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported dataset format: {path}")


def canonicalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    lower_map = {c.lower().strip(): c for c in df.columns}

    aliases = {
        "question": ["question", "instruction", "query"],
        "answer": ["answer", "output", "response", "gold"],
        "difficulty": ["difficulty"],
        "main_topic": ["main topic", "main_topic", "topic", "domain", "subject"],
        "sub_topic": ["sub topic", "sub_topic", "subdomain"],
        "main_format": ["main format", "main_format", "format"],
        "sub_format": ["sub format", "sub_format", "type"],
        "question_number": ["question_number", "question number", "id"],
        "book_name": ["book_name", "book", "book name"],
        "page_number_range": ["page_number_range", "page range", "page_number", "page"],
        "batch_number": ["batch_number", "batch", "batch number"],
        "prompt": ["prompt"],
    }

    for canonical, names in aliases.items():
        for n in names:
            if n in lower_map:
                rename_map[lower_map[n]] = canonical
                break

    out = df.rename(columns=rename_map).copy()

    for col in [
        "question", "answer", "difficulty", "main_topic", "sub_topic",
        "main_format", "sub_format", "question_number", "book_name",
        "page_number_range", "batch_number", "prompt",
    ]:
        if col not in out.columns:
            out[col] = None

    starts, ends = [], []
    for val in out["page_number_range"].tolist():
        s, e = parse_page_range(val)
        starts.append(s)
        ends.append(e)
    out["page_start"] = starts
    out["page_end"] = ends

    out["question"] = out["question"].fillna("").astype(str)
    out["answer"] = out["answer"].fillna("").astype(str)
    out["book_name"] = out["book_name"].fillna("").astype(str)

    return out


def validate_schema(df: pd.DataFrame) -> pd.DataFrame:
    issues: List[Dict[str, Any]] = []
    for i, row in df.iterrows():
        for field in MANDATORY_FIELDS:
            bad = pd.isna(row.get(field)) or str(row.get(field)).strip() == ""
            if bad:
                issues.append({"row_index": int(i), "issue_type": "missing_field", "field": field})
        s, e = row.get("page_start"), row.get("page_end")
        if s is None or e is None:
            issues.append({"row_index": int(i), "issue_type": "invalid_page_range", "field": "page_number_range"})
        elif int(s) <= 0 or int(e) <= 0:
            issues.append({"row_index": int(i), "issue_type": "non_positive_page", "field": "page_number_range"})
    return pd.DataFrame(issues)


def build_book_index(books_dir: Path) -> Dict[str, Path]:
    idx: Dict[str, Path] = {}
    for p in books_dir.glob("*.pdf"):
        idx[p.name.lower().strip()] = p
        idx[p.stem.lower().strip()] = p
    return idx


def resolve_book_path(book_name: str, book_index: Dict[str, Path]) -> Optional[Path]:
    k = (book_name or "").strip().lower()
    if not k:
        return None
    if k in book_index:
        return book_index[k]

    # fuzzy fallback: exact stem/name containment
    for kk, vv in book_index.items():
        if k == kk or k in kk or kk in k:
            return vv
    return None


def extract_page_texts(pdf_path: Path, page_start: int, page_end: int) -> Tuple[str, List[int], int]:
    doc = fitz.open(pdf_path)
    try:
        total_pages = len(doc)
        s = max(1, page_start)
        e = min(total_pages, page_end)
        pages = list(range(s, e + 1))
        chunks = []
        for p in pages:
            t = doc[p - 1].get_text("text") or ""
            chunks.append(f"[Page {p}]\n{t.strip()}")
        text = "\n\n".join(chunks)
        return text, pages, total_pages
    finally:
        doc.close()


def best_sentence_overlap(answer: str, evidence_text: str) -> Tuple[float, str]:
    sentences = sentence_split(evidence_text)
    if not sentences:
        return 0.0, ""
    best_score = 0.0
    best_sent = ""
    for s in sentences:
        score = jaccard_score(answer, s)
        if score > best_score:
            best_score = score
            best_sent = s
    return best_score, best_sent


def split_claims(answer: str) -> List[str]:
    claims = sentence_split(answer)
    if claims:
        return claims[:8]
    return [normalize_text(answer)] if normalize_text(answer) else []


def claim_support_score(answer: str, evidence_text: str) -> float:
    claims = split_claims(answer)
    if not claims:
        return 0.0
    sentences = sentence_split(evidence_text)
    if not sentences:
        return 0.0

    claim_scores = []
    for c in claims:
        claim_scores.append(max((jaccard_score(c, s) for s in sentences), default=0.0))
    return sum(claim_scores) / len(claim_scores)


def contradiction_heuristics(answer: str, evidence_text: str) -> Tuple[bool, List[str]]:
    reasons = []

    ans_nums = set(re.findall(r"\b\d+\b", answer))
    ev_nums = set(re.findall(r"\b\d+\b", evidence_text))
    if ans_nums and ev_nums:
        # If answer contains many numeric values not in evidence, flag mismatch risk
        extra = ans_nums - ev_nums
        if len(extra) >= 2:
            reasons.append("numeric_mismatch")

    ans_lower = answer.lower()
    ev_lower = evidence_text.lower()
    neg_terms = [" not ", " never ", " no ", " cannot ", " false "]
    ans_neg = any(t in f" {ans_lower} " for t in neg_terms)
    ev_neg = any(t in f" {ev_lower} " for t in neg_terms)
    if ans_neg != ev_neg and len(answer) > 20:
        reasons.append("polarity_mismatch")

    return len(reasons) > 0, reasons


def deterministic_grounding(answer: str, evidence_text: str, pages: List[int], cfg: RuntimeConfig) -> DeterministicResult:
    lex, best_sent = best_sentence_overlap(answer, evidence_text)
    claim = claim_support_score(answer, evidence_text)
    contradiction, contradiction_reasons = contradiction_heuristics(answer, evidence_text)

    if contradiction:
        verdict, reason = "fail", "contradiction_detected"
    elif lex >= cfg.lexical_threshold and claim >= cfg.claim_support_threshold:
        verdict, reason = "pass", "strong_support"
    elif lex >= cfg.weak_support_floor or claim >= cfg.weak_support_floor:
        verdict, reason = "uncertain", "weak_support"
    else:
        verdict, reason = "fail", "no_support"

    return DeterministicResult(
        lexical_score=lex,
        claim_support_score=claim,
        contradiction=contradiction,
        contradiction_reasons=contradiction_reasons,
        verdict=verdict,
        reason_code=reason,
        best_support_sentence=best_sent,
        matched_pages=pages,
    )


def build_judge_client(cfg: RuntimeConfig) -> Optional[OpenAI]:
    if not cfg.enable_llm_judge:
        return None
    if not cfg.judge_base_url:
        logger.warning("LLM judge disabled at runtime: missing TEXT_BASE_URL or TEXT_API_KEY.")
        return None
    return OpenAI(base_url=cfg.judge_base_url, api_key=cfg.judge_api_key)


def parse_judge_json(raw: str) -> Optional[Dict[str, Any]]:
    cleaned = re.sub(r"```(?:json)?\s*", "", raw or "").strip().rstrip("`").strip()
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None


def llm_adjudicate(
    client: OpenAI,
    cfg: RuntimeConfig,
    question: str,
    answer: str,
    evidence_text: str,
    page_start: int,
    page_end: int,
) -> JudgeResult:
    system = (
        "You are a strict academic fact-checker. "
        "Judge whether the answer is grounded ONLY in provided source evidence. "
        "Return only JSON."
    )
    user = f"""
Question:
{question}

Proposed Answer:
{answer}

Cited Source Page Range: {page_start}-{page_end}
Evidence Text:
{evidence_text[:cfg.max_evidence_chars]}

Return JSON exactly with keys:
- verdict: supported | contradicted | insufficient
- confidence: number between 0 and 1
- reason_code: short snake_case string
- cited_spans: array of short evidence snippets used
"""

    resp = client.chat.completions.create(
        model=cfg.judge_model,
        temperature=0.0,
        max_tokens=450,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    raw = (resp.choices[0].message.content or "").strip()
    obj = parse_judge_json(raw) or {}

    verdict = str(obj.get("verdict", "insufficient")).strip().lower()
    if verdict not in {"supported", "contradicted", "insufficient"}:
        verdict = "insufficient"

    try:
        confidence = float(obj.get("confidence", 0.0))
    except Exception:
        confidence = 0.0

    reason_code = str(obj.get("reason_code", "judge_unparsed")).strip() or "judge_unparsed"
    cited = obj.get("cited_spans", [])
    if not isinstance(cited, list):
        cited = []

    return JudgeResult(verdict=verdict, confidence=confidence, reason_code=reason_code, cited_spans=cited[:6])


def should_call_judge(det: DeterministicResult, cfg: RuntimeConfig) -> bool:
    if not cfg.enable_llm_judge:
        return False
    if not cfg.judge_on_uncertain_only:
        return True
    return det.verdict == "uncertain"


def merge_verdicts(det: DeterministicResult, judge: Optional[JudgeResult], cfg: RuntimeConfig) -> Tuple[str, str]:
    if det.verdict == "pass":
        return "pass", det.reason_code
    if det.verdict == "fail" and det.reason_code == "contradiction_detected":
        return "fail", det.reason_code

    if judge is None:
        return ("pass", det.reason_code) if det.verdict == "pass" else ("fail", det.reason_code)

    if judge.verdict == "supported" and judge.confidence >= cfg.judge_confidence_threshold:
        return "pass", f"judge_supported::{judge.reason_code}"
    if judge.verdict == "contradicted":
        return "fail", f"judge_contradicted::{judge.reason_code}"
    return "fail", f"judge_insufficient::{judge.reason_code}"


def save_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def validate_dataset(cfg: RuntimeConfig) -> Dict[str, Path]:
    df_raw = load_dataset(cfg.dataset_path)
    df = canonicalize_columns(df_raw)

    schema_issues = validate_schema(df)
    schema_path = cfg.output_dir / "schema_issues.csv"
    schema_issues.to_csv(schema_path, index=False)

    if cfg.max_rows is not None:
        df = df.head(cfg.max_rows).copy()

    book_index = build_book_index(cfg.books_dir)
    judge_client = build_judge_client(cfg)

    evidence_cache: Dict[str, Tuple[str, List[int], int]] = {}
    row_report: List[Dict[str, Any]] = []
    evidence_report: List[Dict[str, Any]] = []
    clean_rows: List[Dict[str, Any]] = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Validating rows"):
        question = str(row.get("question", ""))
        answer = str(row.get("answer", ""))
        book_name = str(row.get("book_name", "")).strip()
        page_start = row.get("page_start")
        page_end = row.get("page_end")

        base_entry = {
            "row_index": int(idx),
            "question": question,
            "answer": answer,
            "difficulty": row.get("difficulty"),
            "main_topic": row.get("main_topic"),
            "sub_topic": row.get("sub_topic"),
            "main_format": row.get("main_format"),
            "sub_format": row.get("sub_format"),
            "question_number": row.get("question_number"),
            "book_name": book_name,
            "page_start": page_start,
            "page_end": page_end,
            "batch_number": row.get("batch_number"),
            "prompt": row.get("prompt"),
        }

        if not question.strip() or not answer.strip() or not book_name or pd.isna(page_start) or pd.isna(page_end):
            out = {
                **base_entry,
                "status": "fail",
                "reason_code": "missing_required_fields",
                "lexical_score": 0.0,
                "claim_support_score": 0.0,
                "judge_verdict": None,
                "judge_confidence": None,
                "matched_pages": [],
                "best_support_sentence": "",
            }
            row_report.append(out)
            continue

        book_path = resolve_book_path(book_name, book_index)
        if book_path is None:
            out = {
                **base_entry,
                "status": "fail",
                "reason_code": "book_not_found",
                "lexical_score": 0.0,
                "claim_support_score": 0.0,
                "judge_verdict": None,
                "judge_confidence": None,
                "matched_pages": [],
                "best_support_sentence": "",
            }
            row_report.append(out)
            continue

        cache_key = f"{book_path}::{int(page_start)}::{int(page_end)}"
        if cache_key not in evidence_cache:
            evidence_cache[cache_key] = extract_page_texts(book_path, int(page_start), int(page_end))

        evidence_text, pages, total_pages = evidence_cache[cache_key]
        if int(page_start) > total_pages:
            out = {
                **base_entry,
                "status": "fail",
                "reason_code": "page_out_of_bounds",
                "lexical_score": 0.0,
                "claim_support_score": 0.0,
                "judge_verdict": None,
                "judge_confidence": None,
                "matched_pages": [],
                "best_support_sentence": "",
            }
            row_report.append(out)
            continue

        det = deterministic_grounding(answer, evidence_text, pages, cfg)

        judge_result: Optional[JudgeResult] = None
        if judge_client is not None and should_call_judge(det, cfg):
            try:
                judge_result = llm_adjudicate(
                    judge_client,
                    cfg,
                    question=question,
                    answer=answer,
                    evidence_text=evidence_text,
                    page_start=int(page_start),
                    page_end=int(page_end),
                )
            except Exception as exc:
                logger.warning("Judge failed for row %s: %s", idx, exc)
                judge_result = JudgeResult(
                    verdict="insufficient",
                    confidence=0.0,
                    reason_code="judge_exception",
                    cited_spans=[],
                )

        status, reason_code = merge_verdicts(det, judge_result, cfg)

        row_entry = {
            **base_entry,
            "status": status,
            "reason_code": reason_code,
            "lexical_score": det.lexical_score,
            "claim_support_score": det.claim_support_score,
            "contradiction": det.contradiction,
            "contradiction_reasons": ";".join(det.contradiction_reasons),
            "matched_pages": ",".join(str(p) for p in det.matched_pages),
            "best_support_sentence": det.best_support_sentence,
            "judge_verdict": judge_result.verdict if judge_result else None,
            "judge_confidence": judge_result.confidence if judge_result else None,
            "judge_reason_code": judge_result.reason_code if judge_result else None,
        }
        row_report.append(row_entry)

        evidence_report.append({
            "row_index": int(idx),
            "book_name": book_name,
            "page_start": int(page_start),
            "page_end": int(page_end),
            "matched_pages": det.matched_pages,
            "best_support_sentence": det.best_support_sentence,
            "judge_cited_spans": judge_result.cited_spans if judge_result else [],
            "evidence_excerpt": evidence_text[:2200],
        })

        if status == "pass":
            clean_rows.append(row.to_dict())

    row_df = pd.DataFrame(row_report)
    row_path_csv = cfg.output_dir / "qa_validation_row_report.csv"
    row_path_jsonl = cfg.output_dir / "qa_validation_row_report.jsonl"
    row_df.to_csv(row_path_csv, index=False)
    save_jsonl(row_path_jsonl, row_report)

    evidence_path_jsonl = cfg.output_dir / "qa_validation_evidence_report.jsonl"
    save_jsonl(evidence_path_jsonl, evidence_report)

    clean_path_jsonl = cfg.output_dir / "qa_validation_clean_export.jsonl"
    save_jsonl(clean_path_jsonl, clean_rows)

    summary = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(cfg.dataset_path),
        "books_dir": str(cfg.books_dir),
        "rows_total": int(len(row_report)),
        "rows_pass": int(sum(1 for r in row_report if r["status"] == "pass")),
        "rows_fail": int(sum(1 for r in row_report if r["status"] == "fail")),
        "pass_rate": float(sum(1 for r in row_report if r["status"] == "pass") / max(1, len(row_report))),
        "reason_counts": pd.Series([r["reason_code"] for r in row_report]).value_counts().to_dict(),
        "output_files": {
            "schema_issues_csv": str(schema_path),
            "row_report_csv": str(row_path_csv),
            "row_report_jsonl": str(row_path_jsonl),
            "evidence_report_jsonl": str(evidence_path_jsonl),
            "clean_export_jsonl": str(clean_path_jsonl),
        },
    }

    summary_path = cfg.output_dir / "qa_validation_summary.json"
    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    logger.info("Validation completed. Pass rate: %.2f%%", summary["pass_rate"] * 100)
    return {
        "schema_issues_csv": schema_path,
        "row_report_csv": row_path_csv,
        "row_report_jsonl": row_path_jsonl,
        "evidence_report_jsonl": evidence_path_jsonl,
        "clean_export_jsonl": clean_path_jsonl,
        "summary_json": summary_path,
    }


logger.info("Functional module loaded.")

2026-04-08 20:26:03,892 | INFO | Functional module loaded.


## 3B. RAGAS Metrics Augmentation

This section adds RAGAS-based grading per row and uses those scores to strengthen hallucination flagging.

Notes:
- Install if needed: `pip install ragas datasets langchain-openai`
- RAGAS is configured to use the same API base URL and API key from `RuntimeConfig`.
- If RAGAS is unavailable or fails for a row, pipeline falls back to deterministic + optional judge behavior.

In [41]:
!pip install ragas datasets langchain-openai

In [42]:
# RAGAS tuning knobs
RAGAS_ENABLED = True
RAGAS_FAIL_FAITHFULNESS = 0.70
RAGAS_WARN_FAITHFULNESS = 0.80
RAGAS_FAIL_RELEVANCY = 0.55
RAGAS_WARN_RELEVANCY = 0.65
RAGAS_EMBED_MODEL = os.getenv("EMBEDDING_LLM_MODEL_NAME", "Qwen/Qwen3-Embedding-0.6B")

# Runtime storage so compute_ragas_metrics can reuse one configured client pair.
RAGAS_RUNTIME: Dict[str, Any] = {
    "llm": None,
    "embeddings": None,
    "configured": False,
    "error": None,
}


def _import_ragas_components() -> Dict[str, Any]:
    components: Dict[str, Any] = {
        "available": False,
        "evaluate": None,
        "Dataset": None,
        "metrics": {},
        "error": None,
    }
    try:
        from ragas import evaluate  # type: ignore
        from datasets import Dataset  # type: ignore
        from ragas.metrics import faithfulness, answer_relevancy  # type: ignore

        metrics = {
            "faithfulness": faithfulness,
            "answer_relevancy": answer_relevancy,
        }

        # Optional metric; availability depends on ragas version.
        try:
            from ragas.metrics import context_precision  # type: ignore
            metrics["context_precision"] = context_precision
        except Exception:
            pass

        components.update(
            {
                "available": True,
                "evaluate": evaluate,
                "Dataset": Dataset,
                "metrics": metrics,
            }
        )
        return components
    except Exception as exc:
        components["error"] = str(exc)
        return components


def configure_ragas_runtime(cfg: RuntimeConfig) -> None:
    """Configure RAGAS to use the same OpenAI-compatible endpoint and key as judge settings."""
    RAGAS_RUNTIME.update({"llm": None, "embeddings": None, "configured": False, "error": None})

    if not RAGAS_ENABLED:
        RAGAS_RUNTIME["error"] = "RAGAS disabled"
        return

    if not cfg.judge_base_url or not cfg.judge_api_key:
        RAGAS_RUNTIME["error"] = "Missing judge_base_url or judge_api_key in RuntimeConfig"
        return

    try:
        # These wrappers let RAGAS use the same OpenAI-compatible backend as your judge.
        from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # type: ignore

        llm = ChatOpenAI(
            model=cfg.judge_model,
            base_url=cfg.judge_base_url,
            api_key=cfg.judge_api_key,
            temperature=0.0,
        )
        embeddings = OpenAIEmbeddings(
            model=RAGAS_EMBED_MODEL,
            base_url=cfg.judge_base_url,
            api_key=cfg.judge_api_key,
        )

        RAGAS_RUNTIME.update(
            {
                "llm": llm,
                "embeddings": embeddings,
                "configured": True,
                "error": None,
            }
        )
    except Exception as exc:
        RAGAS_RUNTIME["error"] = str(exc)


def compute_ragas_metrics(question: str, answer: str, evidence_text: str) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "ragas_available": False,
        "ragas_error": None,
        "faithfulness": None,
        "answer_relevancy": None,
        "context_precision": None,
        "ragas_flag": "not_scored",
    }

    if not RAGAS_ENABLED:
        out["ragas_flag"] = "disabled"
        return out

    comps = _import_ragas_components()
    if not comps["available"]:
        out["ragas_error"] = comps.get("error")
        return out

    evaluate = comps["evaluate"]
    Dataset = comps["Dataset"]
    metrics_map = comps["metrics"]

    contexts = sentence_split(evidence_text)
    if not contexts:
        out["ragas_flag"] = "no_context"
        return out

    contexts = contexts[:25]
    payload = {
        "question": [question],
        "answer": [answer],
        "contexts": [contexts],
    }
    ds = Dataset.from_dict(payload)

    metric_order = [
        name
        for name in ["faithfulness", "answer_relevancy", "context_precision"]
        if name in metrics_map
    ]
    metric_objects = [metrics_map[n] for n in metric_order]

    eval_kwargs: Dict[str, Any] = {"metrics": metric_objects}

    # Force RAGAS to use same backend settings as judge if configured.
    if RAGAS_RUNTIME.get("configured"):
        eval_kwargs["llm"] = RAGAS_RUNTIME.get("llm")
        eval_kwargs["embeddings"] = RAGAS_RUNTIME.get("embeddings")
    elif RAGAS_RUNTIME.get("error"):
        out["ragas_error"] = f"runtime_not_configured: {RAGAS_RUNTIME['error']}"

    try:
        res = evaluate(ds, **eval_kwargs)
    except Exception:
        fallback_names = [n for n in ["faithfulness", "answer_relevancy"] if n in metrics_map]
        if not fallback_names:
            if not out["ragas_error"]:
                out["ragas_error"] = "No compatible RAGAS metrics available"
            return out

        fallback_kwargs: Dict[str, Any] = {"metrics": [metrics_map[n] for n in fallback_names]}
        if RAGAS_RUNTIME.get("configured"):
            fallback_kwargs["llm"] = RAGAS_RUNTIME.get("llm")
            fallback_kwargs["embeddings"] = RAGAS_RUNTIME.get("embeddings")

        try:
            res = evaluate(ds, **fallback_kwargs)
        except Exception as exc:
            msg = str(exc)
            if out["ragas_error"]:
                out["ragas_error"] = f"{out['ragas_error']} | {msg}"
            else:
                out["ragas_error"] = msg
            return out

    try:
        rec = res.to_pandas().iloc[0].to_dict()
    except Exception:
        try:
            rec = dict(res)
        except Exception:
            rec = {}

    out["ragas_available"] = True
    for k in ["faithfulness", "answer_relevancy", "context_precision"]:
        val = rec.get(k)
        if val is not None:
            try:
                out[k] = float(val)
            except Exception:
                out[k] = None

    f = out["faithfulness"]
    r = out["answer_relevancy"]

    if f is not None and f < RAGAS_FAIL_FAITHFULNESS:
        out["ragas_flag"] = "fail_low_faithfulness"
    elif r is not None and r < RAGAS_FAIL_RELEVANCY:
        out["ragas_flag"] = "fail_low_relevancy"
    elif (f is not None and f < RAGAS_WARN_FAITHFULNESS) or (r is not None and r < RAGAS_WARN_RELEVANCY):
        out["ragas_flag"] = "warn_borderline"
    else:
        out["ragas_flag"] = "pass"

    return out


def apply_ragas_flag_to_status(current_status: str, current_reason: str, ragas: Dict[str, Any]) -> Tuple[str, str]:
    flag = ragas.get("ragas_flag", "not_scored")
    if flag.startswith("fail_"):
        return "fail", f"ragas::{flag}"
    if flag == "warn_borderline" and current_status == "pass":
        return "fail", "ragas::warn_borderline"
    return current_status, current_reason


# Override validate_dataset to integrate RAGAS scoring + flagging.
def validate_dataset(cfg: RuntimeConfig) -> Dict[str, Path]:
    df_raw = load_dataset(cfg.dataset_path)
    df = canonicalize_columns(df_raw)

    schema_issues = validate_schema(df)
    schema_path = cfg.output_dir / "schema_issues.csv"
    schema_issues.to_csv(schema_path, index=False)

    if cfg.max_rows is not None:
        df = df.head(cfg.max_rows).copy()

    book_index = build_book_index(cfg.books_dir)
    judge_client = build_judge_client(cfg)
    configure_ragas_runtime(cfg)

    evidence_cache: Dict[str, Tuple[str, List[int], int]] = {}
    row_report: List[Dict[str, Any]] = []
    evidence_report: List[Dict[str, Any]] = []
    clean_rows: List[Dict[str, Any]] = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Validating rows"):
        question = str(row.get("question", ""))
        answer = str(row.get("answer", ""))
        book_name = str(row.get("book_name", "")).strip()
        page_start = row.get("page_start")
        page_end = row.get("page_end")

        base_entry = {
            "row_index": int(idx),
            "question": question,
            "answer": answer,
            "difficulty": row.get("difficulty"),
            "main_topic": row.get("main_topic"),
            "sub_topic": row.get("sub_topic"),
            "main_format": row.get("main_format"),
            "sub_format": row.get("sub_format"),
            "question_number": row.get("question_number"),
            "book_name": book_name,
            "page_start": page_start,
            "page_end": page_end,
            "batch_number": row.get("batch_number"),
            "prompt": row.get("prompt"),
        }

        if not question.strip() or not answer.strip() or not book_name or pd.isna(page_start) or pd.isna(page_end):
            out = {
                **base_entry,
                "status": "fail",
                "reason_code": "missing_required_fields",
                "lexical_score": 0.0,
                "claim_support_score": 0.0,
                "judge_verdict": None,
                "judge_confidence": None,
                "matched_pages": [],
                "best_support_sentence": "",
                "ragas_faithfulness": None,
                "ragas_answer_relevancy": None,
                "ragas_context_precision": None,
                "ragas_flag": "not_scored",
                "ragas_error": None,
            }
            row_report.append(out)
            continue

        book_path = resolve_book_path(book_name, book_index)
        if book_path is None:
            out = {
                **base_entry,
                "status": "fail",
                "reason_code": "book_not_found",
                "lexical_score": 0.0,
                "claim_support_score": 0.0,
                "judge_verdict": None,
                "judge_confidence": None,
                "matched_pages": [],
                "best_support_sentence": "",
                "ragas_faithfulness": None,
                "ragas_answer_relevancy": None,
                "ragas_context_precision": None,
                "ragas_flag": "not_scored",
                "ragas_error": None,
            }
            row_report.append(out)
            continue

        cache_key = f"{book_path}::{int(page_start)}::{int(page_end)}"
        if cache_key not in evidence_cache:
            evidence_cache[cache_key] = extract_page_texts(book_path, int(page_start), int(page_end))

        evidence_text, pages, total_pages = evidence_cache[cache_key]
        if int(page_start) > total_pages:
            out = {
                **base_entry,
                "status": "fail",
                "reason_code": "page_out_of_bounds",
                "lexical_score": 0.0,
                "claim_support_score": 0.0,
                "judge_verdict": None,
                "judge_confidence": None,
                "matched_pages": [],
                "best_support_sentence": "",
                "ragas_faithfulness": None,
                "ragas_answer_relevancy": None,
                "ragas_context_precision": None,
                "ragas_flag": "not_scored",
                "ragas_error": None,
            }
            row_report.append(out)
            continue

        det = deterministic_grounding(answer, evidence_text, pages, cfg)

        judge_result: Optional[JudgeResult] = None
        if judge_client is not None and should_call_judge(det, cfg):
            try:
                judge_result = llm_adjudicate(
                    judge_client,
                    cfg,
                    question=question,
                    answer=answer,
                    evidence_text=evidence_text,
                    page_start=int(page_start),
                    page_end=int(page_end),
                )
            except Exception as exc:
                logger.warning("Judge failed for row %s: %s", idx, exc)
                judge_result = JudgeResult(
                    verdict="insufficient",
                    confidence=0.0,
                    reason_code="judge_exception",
                    cited_spans=[],
                )

        status, reason_code = merge_verdicts(det, judge_result, cfg)

        ragas = compute_ragas_metrics(question=question, answer=answer, evidence_text=evidence_text)
        status, reason_code = apply_ragas_flag_to_status(status, reason_code, ragas)

        row_entry = {
            **base_entry,
            "status": status,
            "reason_code": reason_code,
            "lexical_score": det.lexical_score,
            "claim_support_score": det.claim_support_score,
            "contradiction": det.contradiction,
            "contradiction_reasons": ";".join(det.contradiction_reasons),
            "matched_pages": ",".join(str(p) for p in det.matched_pages),
            "best_support_sentence": det.best_support_sentence,
            "judge_verdict": judge_result.verdict if judge_result else None,
            "judge_confidence": judge_result.confidence if judge_result else None,
            "judge_reason_code": judge_result.reason_code if judge_result else None,
            "ragas_faithfulness": ragas.get("faithfulness"),
            "ragas_answer_relevancy": ragas.get("answer_relevancy"),
            "ragas_context_precision": ragas.get("context_precision"),
            "ragas_flag": ragas.get("ragas_flag"),
            "ragas_error": ragas.get("ragas_error"),
        }
        row_report.append(row_entry)

        evidence_report.append(
            {
                "row_index": int(idx),
                "book_name": book_name,
                "page_start": int(page_start),
                "page_end": int(page_end),
                "matched_pages": det.matched_pages,
                "best_support_sentence": det.best_support_sentence,
                "judge_cited_spans": judge_result.cited_spans if judge_result else [],
                "evidence_excerpt": evidence_text[:2200],
                "ragas_flag": ragas.get("ragas_flag"),
                "ragas_faithfulness": ragas.get("faithfulness"),
                "ragas_answer_relevancy": ragas.get("answer_relevancy"),
            }
        )

        if status == "pass":
            clean_rows.append(row.to_dict())

    row_df = pd.DataFrame(row_report)
    row_path_csv = cfg.output_dir / "qa_validation_row_report.csv"
    row_path_jsonl = cfg.output_dir / "qa_validation_row_report.jsonl"
    row_df.to_csv(row_path_csv, index=False)
    save_jsonl(row_path_jsonl, row_report)

    evidence_path_jsonl = cfg.output_dir / "qa_validation_evidence_report.jsonl"
    save_jsonl(evidence_path_jsonl, evidence_report)

    clean_path_jsonl = cfg.output_dir / "qa_validation_clean_export.jsonl"
    save_jsonl(clean_path_jsonl, clean_rows)

    summary = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(cfg.dataset_path),
        "books_dir": str(cfg.books_dir),
        "rows_total": int(len(row_report)),
        "rows_pass": int(sum(1 for r in row_report if r["status"] == "pass")),
        "rows_fail": int(sum(1 for r in row_report if r["status"] == "fail")),
        "pass_rate": float(sum(1 for r in row_report if r["status"] == "pass") / max(1, len(row_report))),
        "reason_counts": pd.Series([r["reason_code"] for r in row_report]).value_counts().to_dict(),
        "ragas_flag_counts": pd.Series([r.get("ragas_flag", "not_scored") for r in row_report]).value_counts().to_dict(),
        "ragas_means": {
            "faithfulness": float(row_df["ragas_faithfulness"].dropna().mean()) if "ragas_faithfulness" in row_df else None,
            "answer_relevancy": float(row_df["ragas_answer_relevancy"].dropna().mean()) if "ragas_answer_relevancy" in row_df else None,
            "context_precision": float(row_df["ragas_context_precision"].dropna().mean()) if "ragas_context_precision" in row_df else None,
        },
        "output_files": {
            "schema_issues_csv": str(schema_path),
            "row_report_csv": str(row_path_csv),
            "row_report_jsonl": str(row_path_jsonl),
            "evidence_report_jsonl": str(evidence_path_jsonl),
            "clean_export_jsonl": str(clean_path_jsonl),
        },
    }

    summary_path = cfg.output_dir / "qa_validation_summary.json"
    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    logger.info("Validation completed. Pass rate: %.2f%%", summary["pass_rate"] * 100)
    return {
        "schema_issues_csv": schema_path,
        "row_report_csv": row_path_csv,
        "row_report_jsonl": row_path_jsonl,
        "evidence_report_jsonl": evidence_path_jsonl,
        "clean_export_jsonl": clean_path_jsonl,
        "summary_json": summary_path,
    }


logger.info("RAGAS augmentation loaded. validate_dataset now includes shared OpenAI-compatible backend settings.")

2026-04-08 20:26:04,895 | INFO | RAGAS augmentation loaded. validate_dataset now includes shared OpenAI-compatible backend settings.


In [43]:
# Separate OpenAI-compatible embeddings server for RAGAS.
RAGAS_EMBED_MODEL = os.getenv("EMBEDDING_LLM_MODEL_NAME", "Qwen/Qwen3-Embedding-0.6B")
RAGAS_EMBED_BASE_URL = os.getenv("EMBEDDING_LLM_API_URL", "")
RAGAS_EMBED_API_KEY = os.getenv("EMBEDDING_LLM_API_KEY", "")


def configure_ragas_runtime(cfg: RuntimeConfig) -> None:
    """Configure RAGAS with the judge LLM and a separate embeddings backend when provided."""
    RAGAS_RUNTIME.update({"llm": None, "embeddings": None, "configured": False, "error": None})

    if not RAGAS_ENABLED:
        RAGAS_RUNTIME["error"] = "RAGAS disabled"
        return

    if not cfg.judge_base_url:
        RAGAS_RUNTIME["error"] = "Missing judge_base_url or judge_api_key in RuntimeConfig"
        return

    embed_base_url = RAGAS_EMBED_BASE_URL or cfg.judge_base_url
    embed_api_key = RAGAS_EMBED_API_KEY or cfg.judge_api_key
    if not embed_base_url:
        RAGAS_RUNTIME["error"] = "Missing RAGAS_EMBED_BASE_URL or RAGAS_EMBED_API_KEY"
        return

    try:
        from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # type: ignore

        llm = ChatOpenAI(
            model=cfg.judge_model,
            base_url=cfg.judge_base_url,
            api_key=cfg.judge_api_key,
            temperature=0.0,
        )
        embeddings = OpenAIEmbeddings(
            model=RAGAS_EMBED_MODEL,
            base_url=embed_base_url,
            api_key=embed_api_key,
        )

        RAGAS_RUNTIME.update(
            {
                "llm": llm,
                "embeddings": embeddings,
                "configured": True,
                "error": None,
            }
        )
    except Exception as exc:
        RAGAS_RUNTIME["error"] = str(exc)


In [44]:
# Patch: robust CSV export fallback for environments that require escapechar.
import csv

_validate_dataset_original = validate_dataset


def validate_dataset(cfg: RuntimeConfig) -> Dict[str, Path]:
    try:
        return _validate_dataset_original(cfg)
    except Exception as exc:
        if "need to escape, but no escapechar set" not in str(exc):
            raise

        logger.warning(
            "Retrying validation with CSV escape fallback due to writer configuration: %s",
            exc,
        )

        original_to_csv = pd.DataFrame.to_csv

        def _safe_to_csv(self, *args, **kwargs):
            kwargs = dict(kwargs)
            kwargs.setdefault("quoting", csv.QUOTE_MINIMAL)
            kwargs.setdefault("escapechar", "\\")
            return original_to_csv(self, *args, **kwargs)

        pd.DataFrame.to_csv = _safe_to_csv
        try:
            return _validate_dataset_original(cfg)
        finally:
            pd.DataFrame.to_csv = original_to_csv


logger.info("Applied CSV escape fallback wrapper for validate_dataset.")

2026-04-08 20:26:04,929 | INFO | Applied CSV escape fallback wrapper for validate_dataset.


In [45]:
# Patch: make JSONL strict/portable by sanitizing NaN/Inf and non-JSON scalars.
def _json_sanitize(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): _json_sanitize(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_json_sanitize(v) for v in value]
    if isinstance(value, tuple):
        return [_json_sanitize(v) for v in value]
    if isinstance(value, (str, int, bool)) or value is None:
        return value

    if isinstance(value, float):
        if pd.isna(value) or value == float("inf") or value == float("-inf"):
            return None
        return value

    # Handle numpy/pandas scalar wrappers (np.float64, pd.Int64Dtype scalars, etc.).
    if hasattr(value, "item"):
        try:
            return _json_sanitize(value.item())
        except Exception:
            pass

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, Path):
        return str(value)
    return str(value)


def save_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            clean_row = _json_sanitize(r)
            f.write(json.dumps(clean_row, ensure_ascii=False, allow_nan=False) + "\n")


logger.info("Applied strict JSONL serialization patch (NaN/Inf -> null).")

2026-04-08 20:26:04,946 | INFO | Applied strict JSONL serialization patch (NaN/Inf -> null).


In [46]:
# Patch: robust RAGAS runtime with separate OpenAI clients and fallback when embedding API returns empty data.
RAGAS_EMBED_MODEL = os.getenv("EMBEDDING_LLM_MODEL_NAME", "Qwen/Qwen3-Embedding-0.6B")
RAGAS_EMBED_BASE_URL = os.getenv("EMBEDDING_LLM_API_URL", "")
RAGAS_EMBED_API_KEY = os.getenv("EMBEDDING_LLM_API_KEY", "")


def _import_ragas_components() -> Dict[str, Any]:
    components: Dict[str, Any] = {
        "available": False,
        "evaluate": None,
        "Dataset": None,
        "metrics": {},
        "error": None,
    }
    try:
        from ragas import evaluate  # type: ignore
        from datasets import Dataset  # type: ignore

        # Prefer non-deprecated imports first.
        try:
            from ragas.metrics.collections import faithfulness, answer_relevancy  # type: ignore
        except Exception:
            from ragas.metrics import faithfulness, answer_relevancy  # type: ignore

        metrics = {
            "faithfulness": faithfulness,
            "answer_relevancy": answer_relevancy,
        }

        # Optional metric depending on installed ragas version.
        try:
            from ragas.metrics.collections import context_precision  # type: ignore
            metrics["context_precision"] = context_precision
        except Exception:
            try:
                from ragas.metrics import context_precision  # type: ignore
                metrics["context_precision"] = context_precision
            except Exception:
                pass

        components.update(
            {
                "available": True,
                "evaluate": evaluate,
                "Dataset": Dataset,
                "metrics": metrics,
            }
        )
        return components
    except Exception as exc:
        components["error"] = str(exc)
        return components


def configure_ragas_runtime(cfg: RuntimeConfig) -> None:
    """Use separate clients for judge LLM and embeddings backend, with a quick embedding health probe."""
    RAGAS_RUNTIME.update(
        {
            "llm": None,
            "embeddings": None,
            "configured": False,
            "error": None,
            "llm_client": None,
            "embed_client": None,
            "embedding_ok": False,
            "embedding_probe_error": None,
        }
    )

    if not RAGAS_ENABLED:
        RAGAS_RUNTIME["error"] = "RAGAS disabled"
        return

    if not cfg.judge_base_url:
        RAGAS_RUNTIME["error"] = "Missing judge_base_url in RuntimeConfig"
        return

    embed_base_url = RAGAS_EMBED_BASE_URL or cfg.judge_base_url
    embed_api_key = RAGAS_EMBED_API_KEY or cfg.judge_api_key

    if not embed_base_url:
        RAGAS_RUNTIME["error"] = "Missing EMBEDDING_LLM_API_URL and judge_base_url"
        return

    try:
        from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # type: ignore

        llm = ChatOpenAI(
            model=cfg.judge_model,
            base_url=cfg.judge_base_url,
            api_key=cfg.judge_api_key,
            temperature=0.0,
        )
        embeddings = OpenAIEmbeddings(
            model=RAGAS_EMBED_MODEL,
            base_url=embed_base_url,
            api_key=embed_api_key,
        )

        # Dedicated raw clients for diagnostics and explicit endpoint separation.
        llm_client = OpenAI(base_url=cfg.judge_base_url, api_key=cfg.judge_api_key)
        embed_client = OpenAI(base_url=embed_base_url, api_key=embed_api_key)

        RAGAS_RUNTIME.update(
            {
                "llm": llm,
                "embeddings": embeddings,
                "llm_client": llm_client,
                "embed_client": embed_client,
                "configured": True,
                "error": None,
            }
        )

        # Probe embedding endpoint once so failures are handled upfront.
        try:
            probe = embed_client.embeddings.create(model=RAGAS_EMBED_MODEL, input=["healthcheck"])
            data = getattr(probe, "data", None)
            if not data:
                raise ValueError("No embedding data received")
            RAGAS_RUNTIME["embedding_ok"] = True
        except Exception as probe_exc:
            RAGAS_RUNTIME["embedding_ok"] = False
            RAGAS_RUNTIME["embedding_probe_error"] = str(probe_exc)
            logger.warning("RAGAS embedding probe failed: %s", probe_exc)

    except Exception as exc:
        RAGAS_RUNTIME["error"] = str(exc)


def compute_ragas_metrics(question: str, answer: str, evidence_text: str) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "ragas_available": False,
        "ragas_error": None,
        "faithfulness": None,
        "answer_relevancy": None,
        "context_precision": None,
        "ragas_flag": "not_scored",
    }

    if not RAGAS_ENABLED:
        out["ragas_flag"] = "disabled"
        return out

    comps = _import_ragas_components()
    if not comps["available"]:
        out["ragas_error"] = comps.get("error")
        return out

    evaluate = comps["evaluate"]
    Dataset = comps["Dataset"]
    metrics_map = comps["metrics"]

    contexts = sentence_split(evidence_text)
    if not contexts:
        out["ragas_flag"] = "no_context"
        return out

    payload = {
        "question": [question],
        "answer": [answer],
        "contexts": [contexts[:25]],
    }
    ds = Dataset.from_dict(payload)

    # If embedding endpoint is unhealthy, skip embedding-dependent metrics immediately.
    embedding_ok = bool(RAGAS_RUNTIME.get("embedding_ok", False))

    metric_order = ["faithfulness"]
    if embedding_ok:
        for metric_name in ["answer_relevancy", "context_precision"]:
            if metric_name in metrics_map:
                metric_order.append(metric_name)

    metric_objects = [metrics_map[n] for n in metric_order if n in metrics_map]
    if not metric_objects:
        out["ragas_error"] = "No compatible RAGAS metrics available"
        return out

    eval_kwargs: Dict[str, Any] = {"metrics": metric_objects}

    if RAGAS_RUNTIME.get("configured"):
        eval_kwargs["llm"] = RAGAS_RUNTIME.get("llm")
        if embedding_ok:
            eval_kwargs["embeddings"] = RAGAS_RUNTIME.get("embeddings")
    elif RAGAS_RUNTIME.get("error"):
        out["ragas_error"] = f"runtime_not_configured: {RAGAS_RUNTIME['error']}"

    try:
        res = evaluate(ds, **eval_kwargs)
    except Exception as exc:
        msg = str(exc)

        # Fallback: rerun with faithfulness-only when embedding provider fails.
        if "No embedding data received" in msg or "embedding" in msg.lower():
            try:
                faithfulness_only = [metrics_map["faithfulness"]] if "faithfulness" in metrics_map else []
                if not faithfulness_only:
                    out["ragas_error"] = msg
                    return out

                fallback_kwargs: Dict[str, Any] = {"metrics": faithfulness_only}
                if RAGAS_RUNTIME.get("configured"):
                    fallback_kwargs["llm"] = RAGAS_RUNTIME.get("llm")

                res = evaluate(ds, **fallback_kwargs)
                out["ragas_error"] = f"embedding_metric_fallback: {msg}"
            except Exception as fallback_exc:
                out["ragas_error"] = f"{msg} | fallback_failed: {fallback_exc}"
                return out
        else:
            out["ragas_error"] = msg
            return out

    try:
        rec = res.to_pandas().iloc[0].to_dict()
    except Exception:
        try:
            rec = dict(res)
        except Exception:
            rec = {}

    out["ragas_available"] = True
    for k in ["faithfulness", "answer_relevancy", "context_precision"]:
        val = rec.get(k)
        if val is not None:
            try:
                out[k] = float(val)
            except Exception:
                out[k] = None

    f = out["faithfulness"]
    r = out["answer_relevancy"]

    if f is not None and f < RAGAS_FAIL_FAITHFULNESS:
        out["ragas_flag"] = "fail_low_faithfulness"
    elif r is not None and r < RAGAS_FAIL_RELEVANCY:
        out["ragas_flag"] = "fail_low_relevancy"
    elif (f is not None and f < RAGAS_WARN_FAITHFULNESS) or (r is not None and r < RAGAS_WARN_RELEVANCY):
        out["ragas_flag"] = "warn_borderline"
    else:
        out["ragas_flag"] = "pass"

    return out


logger.info("Applied RAGAS patch: separate embedding client + deprecated import fixes + embedding fallback.")

2026-04-08 20:26:04,971 | INFO | Applied RAGAS patch: separate embedding client + deprecated import fixes + embedding fallback.


In [47]:
# Patch override: force embeddings to use EMBEDDING_LLM_API_URL only (no judge URL fallback).
def configure_ragas_runtime(cfg: RuntimeConfig) -> None:
    """Configure separate LLM and embedding clients; embeddings must use EMBEDDING_LLM_API_URL."""
    RAGAS_RUNTIME.update(
        {
            "llm": None,
            "embeddings": None,
            "configured": False,
            "error": None,
            "llm_client": None,
            "embed_client": None,
            "embedding_ok": False,
            "embedding_probe_error": None,
        }
    )

    if not RAGAS_ENABLED:
        RAGAS_RUNTIME["error"] = "RAGAS disabled"
        return

    if not cfg.judge_base_url:
        RAGAS_RUNTIME["error"] = "Missing judge_base_url in RuntimeConfig"
        return

    embed_base_url = (RAGAS_EMBED_BASE_URL or "").strip()
    embed_api_key = (RAGAS_EMBED_API_KEY or cfg.judge_api_key or "").strip()

    if not embed_base_url:
        RAGAS_RUNTIME["error"] = "Missing EMBEDDING_LLM_API_URL. Embeddings will not use judge URL fallback."
        logger.warning(RAGAS_RUNTIME["error"])
        return

    if embed_base_url.rstrip("/") == cfg.judge_base_url.rstrip("/"):
        logger.warning(
            "EMBEDDING_LLM_API_URL matches judge_base_url. Set a dedicated embedding endpoint if intended."
        )

    try:
        from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # type: ignore

        llm = ChatOpenAI(
            model=cfg.judge_model,
            base_url=cfg.judge_base_url,
            api_key=cfg.judge_api_key,
            temperature=0.0,
        )

        embeddings = OpenAIEmbeddings(
            model=RAGAS_EMBED_MODEL,
            base_url=embed_base_url,
            api_key=embed_api_key,
        )

        llm_client = OpenAI(base_url=cfg.judge_base_url, api_key=cfg.judge_api_key)
        embed_client = OpenAI(base_url=embed_base_url, api_key=embed_api_key)

        RAGAS_RUNTIME.update(
            {
                "llm": llm,
                "embeddings": embeddings,
                "llm_client": llm_client,
                "embed_client": embed_client,
                "configured": True,
                "error": None,
            }
        )

        logger.info("RAGAS endpoints | judge=%s | embedding=%s", cfg.judge_base_url, embed_base_url)

        try:
            probe = embed_client.embeddings.create(model=RAGAS_EMBED_MODEL, input=["healthcheck"])
            data = getattr(probe, "data", None)
            if not data:
                raise ValueError("No embedding data received")
            RAGAS_RUNTIME["embedding_ok"] = True
        except Exception as probe_exc:
            RAGAS_RUNTIME["embedding_ok"] = False
            RAGAS_RUNTIME["embedding_probe_error"] = str(probe_exc)
            logger.warning("RAGAS embedding probe failed: %s", probe_exc)

    except Exception as exc:
        RAGAS_RUNTIME["error"] = str(exc)


logger.info("Applied strict embedding URL override for RAGAS runtime.")

2026-04-08 20:26:04,988 | INFO | Applied strict embedding URL override for RAGAS runtime.


In [48]:
# Patch: strengthen LLM judge prompt using full row context from validate_dataset.
import inspect


def _collect_row_context_from_stack() -> Dict[str, Any]:
    """Best-effort capture of row/base_entry fields from validate_dataset caller frame."""
    ctx: Dict[str, Any] = {}
    frame = inspect.currentframe()
    try:
        caller = frame.f_back if frame else None
        if caller is None:
            return ctx

        local_base = caller.f_locals.get("base_entry")
        if isinstance(local_base, dict):
            ctx.update(local_base)

        local_row = caller.f_locals.get("row")
        if local_row is not None and hasattr(local_row, "to_dict"):
            try:
                row_dict = local_row.to_dict()
                if isinstance(row_dict, dict):
                    for k, v in row_dict.items():
                        ctx.setdefault(k, v)
            except Exception:
                pass

        # Add deterministic diagnostics when available.
        det = caller.f_locals.get("det")
        if det is not None:
            try:
                ctx["det_lexical_score"] = getattr(det, "lexical_score", None)
                ctx["det_claim_support_score"] = getattr(det, "claim_support_score", None)
                ctx["det_contradiction"] = getattr(det, "contradiction", None)
                ctx["det_contradiction_reasons"] = getattr(det, "contradiction_reasons", None)
                ctx["det_best_support_sentence"] = getattr(det, "best_support_sentence", None)
                ctx["det_matched_pages"] = getattr(det, "matched_pages", None)
            except Exception:
                pass

        return ctx
    finally:
        del frame


def _normalize_for_prompt(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [_normalize_for_prompt(v) for v in value]
    if isinstance(value, dict):
        return {str(k): _normalize_for_prompt(v) for k, v in value.items()}
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def llm_adjudicate(
    client: OpenAI,
    cfg: RuntimeConfig,
    question: str,
    answer: str,
    evidence_text: str,
    page_start: int,
    page_end: int,
) -> JudgeResult:
    row_context = _collect_row_context_from_stack()

    # Ensure all major CSV/report fields are present in the prompt context when possible.
    prompt_fields = [
        "row_index", "question", "answer", "difficulty", "main_topic", "sub_topic",
        "main_format", "sub_format", "question_number", "book_name", "page_start",
        "page_end", "batch_number", "prompt", "status", "reason_code",
        "lexical_score", "claim_support_score", "contradiction", "contradiction_reasons",
        "matched_pages", "best_support_sentence", "judge_verdict", "judge_confidence",
        "judge_reason_code", "ragas_faithfulness", "ragas_answer_relevancy",
        "ragas_context_precision", "ragas_flag", "ragas_error",
        "det_lexical_score", "det_claim_support_score", "det_contradiction",
        "det_contradiction_reasons", "det_best_support_sentence", "det_matched_pages",
    ]

    normalized_ctx = {}
    for f in prompt_fields:
        if f in row_context:
            normalized_ctx[f] = _normalize_for_prompt(row_context.get(f))

    # Always include current primary fields.
    normalized_ctx["question"] = question
    normalized_ctx["answer"] = answer
    normalized_ctx["page_start"] = page_start
    normalized_ctx["page_end"] = page_end

    system = (
        "You are a strict academic grounding validator. "
        "Judge whether the answer is supported ONLY by the provided evidence text and row metadata. "
        "If evidence is missing or contradictory, do not guess. Return JSON only."
    )

    user = f"""
Validate this QA item using the complete row context and cited evidence.

Row Context JSON:
{json.dumps(normalized_ctx, ensure_ascii=False)}

Evidence Text (from cited pages {page_start}-{page_end}):
{evidence_text[:cfg.max_evidence_chars]}

Decision rubric:
1) supported: all key claims are directly supported by evidence.
2) contradicted: one or more key claims are contradicted by evidence.
3) insufficient: evidence does not contain enough support to verify claims.

Rules:
- Prioritize evidence over metadata labels.
- Penalize fabricated specifics (numbers, names, complexity bounds, line refs) not grounded in evidence.
- Penalize scope drift: answer content outside the cited page span's material.
- If uncertain, choose insufficient.

Return JSON exactly with keys:
- verdict: supported | contradicted | insufficient
- confidence: number between 0 and 1
- reason_code: short snake_case string
- cited_spans: array of short verbatim snippets from evidence
"""

    resp = client.chat.completions.create(
        model=cfg.judge_model,
        temperature=0.0,
        max_tokens=500,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    raw = (resp.choices[0].message.content or "").strip()
    obj = parse_judge_json(raw) or {}

    verdict = str(obj.get("verdict", "insufficient")).strip().lower()
    if verdict not in {"supported", "contradicted", "insufficient"}:
        verdict = "insufficient"

    try:
        confidence = float(obj.get("confidence", 0.0))
    except Exception:
        confidence = 0.0

    reason_code = str(obj.get("reason_code", "judge_unparsed")).strip() or "judge_unparsed"
    cited = obj.get("cited_spans", [])
    if not isinstance(cited, list):
        cited = []

    return JudgeResult(verdict=verdict, confidence=confidence, reason_code=reason_code, cited_spans=cited[:6])


logger.info("Applied robust judge prompt patch using full row context.")

2026-04-08 20:26:05,007 | INFO | Applied robust judge prompt patch using full row context.


In [49]:
# Final override: initialize RAGAS metric objects, log RAGAS errors, and export one consolidated CSV only.
import csv


def _instantiate_metric(metric_candidate: Any) -> Any:
    """Return an initialized metric object whenever metric_candidate is a class/callable factory."""
    if isinstance(metric_candidate, type):
        return metric_candidate()

    # Already-initialized metric object in most ragas versions.
    if hasattr(metric_candidate, "single_turn_ascore") or hasattr(metric_candidate, "ascore"):
        return metric_candidate

    if callable(metric_candidate):
        try:
            return metric_candidate()
        except Exception:
            return metric_candidate

    return metric_candidate


def _import_ragas_components() -> Dict[str, Any]:
    components: Dict[str, Any] = {
        "available": False,
        "evaluate": None,
        "Dataset": None,
        "metrics": {},
        "error": None,
    }

    try:
        from ragas import evaluate  # type: ignore
        from datasets import Dataset  # type: ignore

        # Prefer non-deprecated import path.
        try:
            from ragas.metrics.collections import faithfulness, answer_relevancy  # type: ignore
        except Exception:
            from ragas.metrics import faithfulness, answer_relevancy  # type: ignore

        metrics = {
            "faithfulness": _instantiate_metric(faithfulness),
            "answer_relevancy": _instantiate_metric(answer_relevancy),
        }

        # Optional metric by version.
        try:
            from ragas.metrics.collections import context_precision  # type: ignore
            metrics["context_precision"] = _instantiate_metric(context_precision)
        except Exception:
            try:
                from ragas.metrics import context_precision  # type: ignore
                metrics["context_precision"] = _instantiate_metric(context_precision)
            except Exception:
                pass

        components.update(
            {
                "available": True,
                "evaluate": evaluate,
                "Dataset": Dataset,
                "metrics": metrics,
            }
        )
        return components
    except Exception as exc:
        components["error"] = str(exc)
        return components


def compute_ragas_metrics(question: str, answer: str, evidence_text: str) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "ragas_available": False,
        "ragas_error": None,
        "faithfulness": None,
        "answer_relevancy": None,
        "context_precision": None,
        "ragas_flag": "not_scored",
    }

    if not RAGAS_ENABLED:
        out["ragas_flag"] = "disabled"
        return out

    comps = _import_ragas_components()
    if not comps["available"]:
        out["ragas_error"] = comps.get("error")
        logger.warning("RAGAS unavailable: %s", out["ragas_error"])
        return out

    evaluate = comps["evaluate"]
    Dataset = comps["Dataset"]
    metrics_map = comps["metrics"]

    contexts = sentence_split(evidence_text)
    if not contexts:
        out["ragas_flag"] = "no_context"
        return out

    payload = {
        "question": [question],
        "answer": [answer],
        "contexts": [contexts[:25]],
    }
    ds = Dataset.from_dict(payload)

    embedding_ok = bool(RAGAS_RUNTIME.get("embedding_ok", False))

    metric_order = ["faithfulness"]
    if embedding_ok:
        for metric_name in ["answer_relevancy", "context_precision"]:
            if metric_name in metrics_map:
                metric_order.append(metric_name)

    metric_objects = [metrics_map[n] for n in metric_order if n in metrics_map]
    if not metric_objects:
        out["ragas_error"] = "No compatible RAGAS metrics available"
        logger.warning("RAGAS metric config error: %s", out["ragas_error"])
        return out

    eval_kwargs: Dict[str, Any] = {"metrics": metric_objects}
    if RAGAS_RUNTIME.get("configured"):
        eval_kwargs["llm"] = RAGAS_RUNTIME.get("llm")
        if embedding_ok:
            eval_kwargs["embeddings"] = RAGAS_RUNTIME.get("embeddings")

    try:
        res = evaluate(ds, **eval_kwargs)
    except Exception as exc:
        msg = str(exc)
        if "No embedding data received" in msg or "embedding" in msg.lower():
            try:
                fallback_metrics = [metrics_map["faithfulness"]] if "faithfulness" in metrics_map else []
                if not fallback_metrics:
                    out["ragas_error"] = msg
                    logger.warning("RAGAS failure (no fallback metric): %s", msg)
                    return out

                fallback_kwargs: Dict[str, Any] = {"metrics": fallback_metrics}
                if RAGAS_RUNTIME.get("configured"):
                    fallback_kwargs["llm"] = RAGAS_RUNTIME.get("llm")

                res = evaluate(ds, **fallback_kwargs)
                out["ragas_error"] = f"embedding_metric_fallback: {msg}"
                logger.warning("RAGAS embedding fallback used: %s", msg)
            except Exception as fallback_exc:
                out["ragas_error"] = f"{msg} | fallback_failed: {fallback_exc}"
                logger.warning("RAGAS fallback failed: %s", out["ragas_error"])
                return out
        else:
            out["ragas_error"] = msg
            logger.warning("RAGAS evaluate failed: %s", msg)
            return out

    try:
        rec = res.to_pandas().iloc[0].to_dict()
    except Exception:
        try:
            rec = dict(res)
        except Exception:
            rec = {}

    out["ragas_available"] = True
    for k in ["faithfulness", "answer_relevancy", "context_precision"]:
        val = rec.get(k)
        if val is not None:
            try:
                out[k] = float(val)
            except Exception:
                out[k] = None

    f = out["faithfulness"]
    r = out["answer_relevancy"]

    if f is not None and f < RAGAS_FAIL_FAITHFULNESS:
        out["ragas_flag"] = "fail_low_faithfulness"
    elif r is not None and r < RAGAS_FAIL_RELEVANCY:
        out["ragas_flag"] = "fail_low_relevancy"
    elif (f is not None and f < RAGAS_WARN_FAITHFULNESS) or (r is not None and r < RAGAS_WARN_RELEVANCY):
        out["ragas_flag"] = "warn_borderline"
    else:
        out["ragas_flag"] = "pass"

    return out


def validate_dataset(cfg: RuntimeConfig) -> Dict[str, Path]:
    """Run validation and export one consolidated CSV only.

    Exported CSV includes:
    - all canonical/original dataset fields from each row
    - deterministic + judge + RAGAS metric outputs
    - page context used for validation
    """
    df_raw = load_dataset(cfg.dataset_path)
    df = canonicalize_columns(df_raw)

    if cfg.max_rows is not None:
        df = df.head(cfg.max_rows).copy()

    book_index = build_book_index(cfg.books_dir)
    judge_client = build_judge_client(cfg)
    configure_ragas_runtime(cfg)

    evidence_cache: Dict[str, Tuple[str, List[int], int]] = {}
    final_rows: List[Dict[str, Any]] = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Validating rows"):
        row_dict = row.to_dict()
        question = str(row.get("question", ""))
        answer = str(row.get("answer", ""))
        book_name = str(row.get("book_name", "")).strip()
        page_start = row.get("page_start")
        page_end = row.get("page_end")

        enriched = {
            **row_dict,
            "row_index": int(idx),
            "status": "fail",
            "reason_code": "unprocessed",
            "lexical_score": 0.0,
            "claim_support_score": 0.0,
            "contradiction": False,
            "contradiction_reasons": "",
            "best_support_sentence": "",
            "matched_pages": "",
            "page_range_context": "",
            "evidence_excerpt": "",
            "judge_verdict": None,
            "judge_confidence": None,
            "judge_reason_code": None,
            "ragas_faithfulness": None,
            "ragas_answer_relevancy": None,
            "ragas_context_precision": None,
            "ragas_flag": "not_scored",
        }

        if not question.strip() or not answer.strip() or not book_name or pd.isna(page_start) or pd.isna(page_end):
            enriched["reason_code"] = "missing_required_fields"
            final_rows.append(enriched)
            continue

        book_path = resolve_book_path(book_name, book_index)
        if book_path is None:
            enriched["reason_code"] = "book_not_found"
            final_rows.append(enriched)
            continue

        cache_key = f"{book_path}::{int(page_start)}::{int(page_end)}"
        if cache_key not in evidence_cache:
            evidence_cache[cache_key] = extract_page_texts(book_path, int(page_start), int(page_end))

        evidence_text, pages, total_pages = evidence_cache[cache_key]
        if int(page_start) > total_pages:
            enriched["reason_code"] = "page_out_of_bounds"
            final_rows.append(enriched)
            continue

        det = deterministic_grounding(answer, evidence_text, pages, cfg)

        judge_result: Optional[JudgeResult] = None
        if judge_client is not None and should_call_judge(det, cfg):
            try:
                judge_result = llm_adjudicate(
                    judge_client,
                    cfg,
                    question=question,
                    answer=answer,
                    evidence_text=evidence_text,
                    page_start=int(page_start),
                    page_end=int(page_end),
                )
            except Exception as exc:
                logger.warning("Judge failed for row %s: %s", idx, exc)
                judge_result = JudgeResult(
                    verdict="insufficient",
                    confidence=0.0,
                    reason_code="judge_exception",
                    cited_spans=[],
                )

        status, reason_code = merge_verdicts(det, judge_result, cfg)

        ragas = compute_ragas_metrics(question=question, answer=answer, evidence_text=evidence_text)
        if ragas.get("ragas_error"):
            logger.warning("RAGAS row %s: %s", idx, ragas.get("ragas_error"))

        status, reason_code = apply_ragas_flag_to_status(status, reason_code, ragas)

        matched_pages_text = ",".join(str(p) for p in det.matched_pages)
        page_range_context = f"{min(det.matched_pages)}-{max(det.matched_pages)}" if det.matched_pages else ""

        enriched.update(
            {
                "status": status,
                "reason_code": reason_code,
                "lexical_score": det.lexical_score,
                "claim_support_score": det.claim_support_score,
                "contradiction": det.contradiction,
                "contradiction_reasons": ";".join(det.contradiction_reasons),
                "best_support_sentence": det.best_support_sentence,
                "matched_pages": matched_pages_text,
                "page_range_context": page_range_context,
                "evidence_excerpt": evidence_text[:2200],
                "judge_verdict": judge_result.verdict if judge_result else None,
                "judge_confidence": judge_result.confidence if judge_result else None,
                "judge_reason_code": judge_result.reason_code if judge_result else None,
                "ragas_faithfulness": ragas.get("faithfulness"),
                "ragas_answer_relevancy": ragas.get("answer_relevancy"),
                "ragas_context_precision": ragas.get("context_precision"),
                "ragas_flag": ragas.get("ragas_flag"),
            }
        )

        final_rows.append(enriched)

    final_df = pd.DataFrame(final_rows)

    # Remove ragas_error from exported CSV as requested; errors are logged only.
    if "ragas_error" in final_df.columns:
        final_df = final_df.drop(columns=["ragas_error"])

    out_csv = cfg.output_dir / "qa_validation_single_export.csv"
    final_df.to_csv(out_csv, index=False, quoting=csv.QUOTE_MINIMAL, escapechar="\\")

    pass_rate = float((final_df["status"] == "pass").mean()) if len(final_df) else 0.0
    logger.info("Validation completed. Rows=%s | Pass rate=%.2f%%", len(final_df), pass_rate * 100)
    logger.info("Single export written: %s", out_csv)

    return {"single_export_csv": out_csv}


logger.info("Applied final RAGAS + single CSV export override.")

2026-04-08 20:26:05,026 | INFO | Applied final RAGAS + single CSV export override.


In [ ]:
# Hotfix: robust RAGAS metric initialization + one-time disable to stop repeated metric-object errors.
RAGAS_RUNTIME.setdefault("metric_init_ok", None)
RAGAS_RUNTIME.setdefault("metric_init_error", None)
RAGAS_RUNTIME.setdefault("ragas_disabled_reason", None)


def _safe_import_metric_classes() -> Dict[str, Any]:
    """Locate metric classes across ragas versions and instantiate them."""
    imported: Dict[str, Any] = {}

    # Candidate symbols used across ragas releases.
    candidates = {
        "faithfulness": ["Faithfulness", "faithfulness"],
        "answer_relevancy": ["AnswerRelevancy", "AnswerRelevance", "answer_relevancy"],
        "context_precision": ["ContextPrecision", "context_precision"],
    }

    modules = []
    try:
        import ragas.metrics.collections as m_col  # type: ignore
        modules.append(m_col)
    except Exception:
        pass
    try:
        import ragas.metrics as m_root  # type: ignore
        modules.append(m_root)
    except Exception:
        pass

    def _build(obj: Any) -> Any:
        # If class, instantiate.
        if isinstance(obj, type):
            return obj()
        # If callable factory, try call once.
        if callable(obj) and not hasattr(obj, "single_turn_ascore") and not hasattr(obj, "ascore"):
            try:
                return obj()
            except Exception:
                return obj
        return obj

    for key, names in candidates.items():
        picked = None
        for mod in modules:
            for name in names:
                if hasattr(mod, name):
                    picked = getattr(mod, name)
                    break
            if picked is not None:
                break

        if picked is not None:
            try:
                imported[key] = _build(picked)
            except Exception:
                pass

    return imported


def _validate_metric_objects(metrics: List[Any]) -> Tuple[bool, str]:
    for m in metrics:
        # ragas metric instances usually expose one of these methods.
        if hasattr(m, "single_turn_ascore") or hasattr(m, "ascore"):
            continue
        # Some implementations expose a run-like callable interface.
        if hasattr(m, "score"):
            continue
        return False, f"Invalid metric object type: {type(m)}"
    return True, ""


def _import_ragas_components() -> Dict[str, Any]:
    components: Dict[str, Any] = {
        "available": False,
        "evaluate": None,
        "Dataset": None,
        "metrics": {},
        "error": None,
    }

    try:
        from ragas import evaluate  # type: ignore
        from datasets import Dataset  # type: ignore

        metrics = _safe_import_metric_classes()
        if "faithfulness" not in metrics:
            raise RuntimeError("Could not initialize Faithfulness metric object from installed ragas version")

        metric_objs = [metrics[k] for k in ["faithfulness", "answer_relevancy", "context_precision"] if k in metrics]
        ok, reason = _validate_metric_objects(metric_objs)
        if not ok:
            raise RuntimeError(f"Metric initialization failed: {reason}")

        components.update(
            {
                "available": True,
                "evaluate": evaluate,
                "Dataset": Dataset,
                "metrics": metrics,
            }
        )
        return components

    except Exception as exc:
        components["error"] = str(exc)
        return components


def compute_ragas_metrics(question: str, answer: str, evidence_text: str) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "ragas_available": False,
        "ragas_error": None,
        "faithfulness": None,
        "answer_relevancy": None,
        "context_precision": None,
        "ragas_flag": "not_scored",
    }

    if not RAGAS_ENABLED:
        out["ragas_flag"] = "disabled"
        return out

    # If previously disabled due to initialization issue, skip quietly.
    if RAGAS_RUNTIME.get("ragas_disabled_reason"):
        out["ragas_flag"] = "disabled"
        return out

    comps = _import_ragas_components()
    if not comps["available"]:
        out["ragas_error"] = comps.get("error")
        RAGAS_RUNTIME["ragas_disabled_reason"] = f"metric_init_or_import_error: {out['ragas_error']}"
        logger.warning("Disabling RAGAS for remaining rows: %s", RAGAS_RUNTIME["ragas_disabled_reason"])
        out["ragas_flag"] = "disabled"
        return out

    evaluate = comps["evaluate"]
    Dataset = comps["Dataset"]
    metrics_map = comps["metrics"]

    contexts = sentence_split(evidence_text)
    if not contexts:
        out["ragas_flag"] = "no_context"
        return out

    ds = Dataset.from_dict(
        {
            "question": [question],
            "answer": [answer],
            "contexts": [contexts[:25]],
        }
    )

    embedding_ok = bool(RAGAS_RUNTIME.get("embedding_ok", False))

    metric_order = ["faithfulness"]
    if embedding_ok:
        for metric_name in ["answer_relevancy", "context_precision"]:
            if metric_name in metrics_map:
                metric_order.append(metric_name)

    metric_objects = [metrics_map[n] for n in metric_order if n in metrics_map]
    ok, reason = _validate_metric_objects(metric_objects)
    if not ok:
        out["ragas_error"] = f"pre_evaluate_metric_validation_failed: {reason}"
        RAGAS_RUNTIME["ragas_disabled_reason"] = out["ragas_error"]
        logger.warning("Disabling RAGAS for remaining rows: %s", out["ragas_error"])
        out["ragas_flag"] = "disabled"
        return out

    eval_kwargs: Dict[str, Any] = {"metrics": metric_objects}
    if RAGAS_RUNTIME.get("configured"):
        eval_kwargs["llm"] = RAGAS_RUNTIME.get("llm")
        if embedding_ok:
            eval_kwargs["embeddings"] = RAGAS_RUNTIME.get("embeddings")

    try:
        res = evaluate(ds, **eval_kwargs)
    except Exception as exc:
        msg = str(exc)

        # If metric object init requirement still appears, disable ragas once to avoid row spam.
        if "initialised metric objects" in msg.lower() or "initialized metric objects" in msg.lower():
            out["ragas_error"] = msg
            RAGAS_RUNTIME["ragas_disabled_reason"] = f"evaluate_metric_object_error: {msg}"
            logger.warning("Disabling RAGAS for remaining rows: %s", RAGAS_RUNTIME["ragas_disabled_reason"])
            out["ragas_flag"] = "disabled"
            return out

        if "No embedding data received" in msg or "embedding" in msg.lower():
            try:
                fallback_metrics = [metrics_map["faithfulness"]] if "faithfulness" in metrics_map else []
                ok_fb, reason_fb = _validate_metric_objects(fallback_metrics)
                if not fallback_metrics or not ok_fb:
                    out["ragas_error"] = msg if fallback_metrics else f"{msg} | no_fallback_metric"
                    return out

                fallback_kwargs: Dict[str, Any] = {"metrics": fallback_metrics}
                if RAGAS_RUNTIME.get("configured"):
                    fallback_kwargs["llm"] = RAGAS_RUNTIME.get("llm")

                res = evaluate(ds, **fallback_kwargs)
                out["ragas_error"] = f"embedding_metric_fallback: {msg}"
            except Exception as fallback_exc:
                out["ragas_error"] = f"{msg} | fallback_failed: {fallback_exc}"
                return out
        else:
            out["ragas_error"] = msg
            return out

    try:
        rec = res.to_pandas().iloc[0].to_dict()
    except Exception:
        try:
            rec = dict(res)
        except Exception:
            rec = {}

    out["ragas_available"] = True
    for k in ["faithfulness", "answer_relevancy", "context_precision"]:
        val = rec.get(k)
        if val is not None:
            try:
                out[k] = float(val)
            except Exception:
                out[k] = None

    f = out["faithfulness"]
    r = out["answer_relevancy"]

    if f is not None and f < RAGAS_FAIL_FAITHFULNESS:
        out["ragas_flag"] = "fail_low_faithfulness"
    elif r is not None and r < RAGAS_FAIL_RELEVANCY:
        out["ragas_flag"] = "fail_low_relevancy"
    elif (f is not None and f < RAGAS_WARN_FAITHFULNESS) or (r is not None and r < RAGAS_WARN_RELEVANCY):
        out["ragas_flag"] = "warn_borderline"
    else:
        out["ragas_flag"] = "pass"

    return out


logger.info("Applied RAGAS metric-object hotfix with one-time disable on init errors.")

## 4. Add Configuration and Environment Settings

Set your dataset path and validation profile here. Keep `enable_llm_judge=True` for hybrid mode.

In [50]:
# Replace this with your real file when available.
# DATASET_PATH = ROOT / "outputs" / "synthetic_with_page_ranges.jsonl"
DATASET_PATH = Path("/media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/datasets/training/clrs_dataset_all_chunks.csv")

cfg = RuntimeConfig(
    dataset_path=DATASET_PATH,
    books_dir=DEFAULT_BOOKS_DIR,
    output_dir=DEFAULT_OUTPUT_DIR,
    lexical_threshold=0.22,
    claim_support_threshold=0.18,
    weak_support_floor=0.12,
    enable_llm_judge=True,
    judge_on_uncertain_only=True,
    judge_confidence_threshold=0.72,
    max_rows=None,  # set to e.g. 100 for quick smoke run
)

print("Config loaded:")
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()}, indent=2))

Config loaded:
{
  "dataset_path": "/media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/datasets/training/clrs_dataset_all_chunks.csv",
  "books_dir": "/media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/datasets/training/Books",
  "output_dir": "/media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/outputs/qa_validation",
  "checkpoint_path": "/media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/outputs/qa_validation/validation_checkpoint.json",
  "lexical_threshold": 0.22,
  "claim_support_threshold": 0.18,
  "weak_support_floor": 0.12,
  "enable_llm_judge": true,
  "judge_on_uncertain_only": true,
  "judge_model": "openai/gpt-oss-20b",
  "judge_base_url": "https://llm.ipropel.co.in/v1",
  "judge_api_key": "",
  "judge_confidence_threshold": 0.72,
  "max_evidence_chars": 14000,
  "max_rows": null
}


## 5. Write Basic Unit Tests

Quick unit checks for parsing, schema normalization, and deterministic support scoring.

In [51]:
# parse_page_range tests
assert parse_page_range("(10 - 20)") == (10, 20)
assert parse_page_range("20-10") == (10, 20)
assert parse_page_range("15") == (15, 15)
assert parse_page_range(None) == (None, None)

# canonicalize_columns tests
mini = pd.DataFrame([
    {
        "Question": "What is a binary tree?",
        "Answer": "A hierarchical structure where each node has at most 2 children.",
        "Book_name": "DSA - 1.pdf",
        "Page_number_range": "10-12",
    }
])
canon = canonicalize_columns(mini)
assert "question" in canon.columns and "answer" in canon.columns
assert canon.loc[0, "page_start"] == 10 and canon.loc[0, "page_end"] == 12

# deterministic grounding tests
ev = "[Page 10] A binary tree is a hierarchical structure where each node has at most two children."
det = deterministic_grounding(
    answer="A binary tree is hierarchical and each node has at most two children.",
    evidence_text=ev,
    pages=[10],
    cfg=RuntimeConfig(dataset_path=Path("dummy.jsonl"), enable_llm_judge=False),
)
assert det.verdict in {"pass", "uncertain"}

print("Basic tests passed.")

Basic tests passed.


## 6. Run the Code and Inspect Output

Run the validator and inspect top failures. If your dataset file path is wrong, update `DATASET_PATH` in Section 4 first.

In [52]:
if not cfg.dataset_path.exists():
    print(f"Dataset file not found: {cfg.dataset_path}")
    print("Update DATASET_PATH in Section 4, then rerun this cell.")
else:
    outputs = validate_dataset(cfg)
    print("\nValidation artifacts:")
    for k, v in outputs.items():
        print(f"- {k}: {v}")

2026-04-08 20:26:05,104 | INFO | RAGAS endpoints | judge=https://llm.ipropel.co.in/v1 | embedding=https://embedding-llm.ipropel.co.in/v1
2026-04-08 20:26:05,926 | INFO | HTTP Request: POST https://embedding-llm.ipropel.co.in/v1/embeddings "HTTP/1.1 200 OK"


Validating rows:   0%|          | 0/69 [00:00<?, ?it/s]

2026-04-08 20:26:07,340 | WARNING | RAGAS evaluate failed: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]
2026-04-08 20:26:07,341 | WARNING | RAGAS row 0: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]
2026-04-08 20:26:07,820 | WARNING | RAGAS evaluate failed: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]
2026-04-08 20:26:07,821 | WARNING | RAGAS row 1: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]
2026-04-08 20:26:08,385 | WARNING | RAGAS evaluate failed: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]
2026-04-08 20:26:08,386 | WARNING | RAGAS row 2: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]
2026-04-08 20:26:09,307 | WARNING | RAGAS evaluate failed: All metrics must be initialised metric objects, e.g: metrics=[BleuScore


Validation artifacts:
- single_export_csv: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/outputs/qa_validation/qa_validation_single_export.csv


In [53]:
summary_path = cfg.output_dir / "qa_validation_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("\nSummary:")
    print(json.dumps(summary, indent=2, ensure_ascii=False))

    row_csv = Path(summary["output_files"]["row_report_csv"])
    if row_csv.exists():
        report_df = pd.read_csv(row_csv)
        print("\nTop failure reasons:")
        print(report_df["reason_code"].value_counts().head(10))
        print("\nSample failed rows:")
        display_cols = [
            "row_index", "book_name", "page_start", "page_end", "reason_code",
            "lexical_score", "claim_support_score", "best_support_sentence",
        ]
        print(report_df[report_df["status"] == "fail"][display_cols].head(10).to_string(index=False))
else:
    print("No summary file yet. Run the previous cell first.")

No summary file yet. Run the previous cell first.


## 7. Visualize Results + Export Question-Context File

This section reads saved validation artifacts from disk, builds richer visual summaries, and exports a **single consolidated JSONL** with question, answer, page range, and supporting context snippet.

In [54]:
import matplotlib.pyplot as plt


def load_jsonl_df(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()

    rows = []
    bad_lines = 0
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad_lines += 1

    if bad_lines:
        print(f"Warning: skipped {bad_lines} invalid JSONL lines in {path.name}")
    return pd.DataFrame(rows)


output_dir = cfg.output_dir if "cfg" in globals() else (Path.cwd().resolve() / "outputs" / "qa_validation")
row_csv_path = output_dir / "qa_validation_row_report.csv"
row_jsonl_path = output_dir / "qa_validation_row_report.jsonl"
evidence_jsonl_path = output_dir / "qa_validation_evidence_report.jsonl"

report_df = pd.read_csv(row_csv_path) if row_csv_path.exists() else load_jsonl_df(row_jsonl_path)
evidence_df = load_jsonl_df(evidence_jsonl_path)

if report_df.empty:
    print(f"No report data found in: {output_dir}")
else:
    # Normalize numeric fields for plotting.
    for col in ["lexical_score", "claim_support_score", "judge_confidence", "ragas_faithfulness", "ragas_answer_relevancy"]:
        if col in report_df.columns:
            report_df[col] = pd.to_numeric(report_df[col], errors="coerce")

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    # 1) Status distribution.
    status_counts = report_df["status"].fillna("unknown").value_counts()
    axes[0, 0].bar(status_counts.index.astype(str), status_counts.values)
    axes[0, 0].set_title("Validation Status Distribution")
    axes[0, 0].set_xlabel("status")
    axes[0, 0].set_ylabel("rows")

    # 2) Top failure reasons.
    reason_counts = report_df["reason_code"].fillna("unknown").value_counts().head(10)
    axes[0, 1].barh(reason_counts.index.astype(str), reason_counts.values)
    axes[0, 1].set_title("Top 10 Reason Codes")
    axes[0, 1].set_xlabel("count")

    # 3) Score distribution.
    if {"lexical_score", "claim_support_score"}.issubset(report_df.columns):
        axes[1, 0].hist(report_df["lexical_score"].dropna(), bins=25, alpha=0.7, label="lexical")
        axes[1, 0].hist(report_df["claim_support_score"].dropna(), bins=25, alpha=0.7, label="claim_support")
        axes[1, 0].legend()
        axes[1, 0].set_title("Deterministic Score Distributions")
        axes[1, 0].set_xlabel("score")
        axes[1, 0].set_ylabel("frequency")

    # 4) Pass rate by top books.
    if "book_name" in report_df.columns and "status" in report_df.columns:
        top_books = report_df["book_name"].fillna("unknown").value_counts().head(12).index
        pb = report_df[report_df["book_name"].isin(top_books)].copy()
        pb["is_pass"] = (pb["status"] == "pass").astype(int)
        pass_rate = pb.groupby("book_name", as_index=False)["is_pass"].mean().sort_values("is_pass", ascending=False)

        axes[1, 1].barh(pass_rate["book_name"].astype(str), pass_rate["is_pass"])
        axes[1, 1].set_xlim(0, 1)
        axes[1, 1].set_title("Pass Rate by Book (Top 12 by Volume)")
        axes[1, 1].set_xlabel("pass rate")

    plt.tight_layout()
    plt.show()

    # Optional scatter for quick separability view.
    if {"lexical_score", "claim_support_score", "status"}.issubset(report_df.columns):
        color_map = {"pass": "green", "fail": "red", "uncertain": "orange"}
        colors = report_df["status"].map(color_map).fillna("gray")
        plt.figure(figsize=(8, 6))
        plt.scatter(
            report_df["lexical_score"],
            report_df["claim_support_score"],
            c=colors,
            alpha=0.6,
            s=12,
        )
        plt.title("Lexical vs Claim Support Score")
        plt.xlabel("lexical_score")
        plt.ylabel("claim_support_score")
        plt.show()

    # Build single merged export with page-range context.
    if not evidence_df.empty and "row_index" in evidence_df.columns:
        merged = report_df.merge(
            evidence_df[[
                c for c in [
                    "row_index",
                    "matched_pages",
                    "evidence_excerpt",
                    "best_support_sentence",
                    "page_start",
                    "page_end",
                ]
                if c in evidence_df.columns
            ]],
            on="row_index",
            how="left",
            suffixes=("", "_ev"),
        )
    else:
        merged = report_df.copy()

    def _context_span(row: pd.Series) -> str:
        ps = row.get("page_start")
        pe = row.get("page_end")
        if pd.notna(ps) and pd.notna(pe):
            return f"{int(ps)}-{int(pe)}"

        mp = row.get("matched_pages")
        if isinstance(mp, list) and mp:
            return f"{int(min(mp))}-{int(max(mp))}"
        return ""

    merged["page_range_context"] = merged.apply(_context_span, axis=1)

    consolidated_cols = [
        "row_index",
        "question",
        "answer",
        "book_name",
        "page_start",
        "page_end",
        "page_range_context",
        "matched_pages",
        "best_support_sentence",
        "evidence_excerpt",
        "status",
        "reason_code",
        "lexical_score",
        "claim_support_score",
        "judge_verdict",
        "judge_confidence",
        "ragas_faithfulness",
        "ragas_answer_relevancy",
        "ragas_context_precision",
        "ragas_flag",
    ]
    consolidated_cols = [c for c in consolidated_cols if c in merged.columns]
    consolidated_df = merged[consolidated_cols].copy()

    out_path = output_dir / "qa_validation_questions_with_page_context.jsonl"

    def _sanitize_json_value(v):
        if isinstance(v, dict):
            return {str(k): _sanitize_json_value(x) for k, x in v.items()}
        if isinstance(v, list):
            return [_sanitize_json_value(x) for x in v]
        if isinstance(v, tuple):
            return [_sanitize_json_value(x) for x in v]
        if isinstance(v, float) and (pd.isna(v) or v == float("inf") or v == float("-inf")):
            return None
        try:
            if pd.isna(v):
                return None
        except Exception:
            pass
        if isinstance(v, Path):
            return str(v)
        return v

    with out_path.open("w", encoding="utf-8") as f:
        for rec in consolidated_df.to_dict(orient="records"):
            safe_rec = {k: _sanitize_json_value(v) for k, v in rec.items()}
            f.write(json.dumps(safe_rec, ensure_ascii=False, allow_nan=False) + "\n")

    print(f"\nCreated consolidated file: {out_path}")
    print(f"Rows written: {len(consolidated_df)}")
    print("\nSample rows:")
    print(consolidated_df.head(3).to_string(index=False))

No report data found in: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/outputs/qa_validation
